# Climate Indices Pipeline — Core Run (I/O → Validate → Clean → Units → Indices)

Uses the `climate_indices` package (in this same folder) built from your framework spec.
Scope for this phase: **validation, cleaning, and all ETCCDI indices computable from
tasmax + pr** (no tasmin, no SPEI, no shapefile clipping — those come in a later phase).

**Package layout:**
```
climate_indices/
    io_utils.py        read_dataset()
    validate.py         validate_dataset(), get_physical_range()
    clean.py             clean_dataset(), mask_invalid(), interpolate_gaps()
    units.py             convert_units(), convert_dataset_units()
    indices_temp.py       compute_temperature_indices()   -> TXx, TXn, TX90p, TX10p, WSDI, SU, ID
    indices_precip.py     compute_precipitation_indices()  -> RX1day, RX5day, PRCPTOT, SDII,
                                                               R10mm, R20mm, CDD, CWD,
                                                               R95p, R99p, P95D, P99D
```

Every module was unit-tested against synthetic data before this notebook was written —
including catching and fixing a real bug (a unit-unaware physical-range check that nuked
all values to NaN after Kelvin→Celsius conversion). Worth knowing that history if you hit
something unexpected: the range-check logic in `validate.py`/`clean.py` is unit-aware, keyed
off the array's *current* `units` attribute — if you skip the `units` conversion step or
strip that attribute, range checks silently become no-ops (by design — better to skip a
check than mask blind).


In [ ]:
import sys
sys.path.insert(0, ".")   # so `climate_indices` package is importable from this folder

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

import climate_indices as ci
import xarray as xr


## 1. Read (lazy, Dask-backed)

In [ ]:
tasmax_path = "../data/ACCESS-ESM1-5/tasmax"   # recursively finds files in historical/ AND ssp126/
pr_path     = "../data/ACCESS-ESM1-5/pr"

ds_tasmax = ci.io_utils.read_dataset(tasmax_path, variable="tasmax", chunks={"time": 365})
ds_pr     = ci.io_utils.read_dataset(pr_path, variable="pr", chunks={"time": 365})

ds_tasmax


## 2. Validate

Run once with `sample_for_expensive_checks=True` (cheap, fast) first to catch obvious problems.

In [ ]:
report_tasmax = ci.validate.validate_dataset(ds_tasmax, "tasmax")
print(report_tasmax.summary())


In [ ]:
report_pr = ci.validate.validate_dataset(ds_pr, "pr")
print(report_pr.summary())


**Read the warnings/errors above before continuing.** If either report has entries under
`[errors]`, stop and fix the underlying file/coordinates first — everything downstream
assumes clean `time`/`lat`/`lon` dims.

Once you're confident in your files, re-run validation with the full (non-sampled) pass —
slower, but authoritative:
```python
report_tasmax_full = ci.validate.validate_dataset(ds_tasmax, "tasmax", sample_for_expensive_checks=False)
```


## 3. Convert units

In [ ]:
ds_tasmax = ci.units.convert_dataset_units(ds_tasmax, {"tasmax": "degC"})
ds_pr     = ci.units.convert_dataset_units(ds_pr, {"pr": "mm/day"})

print("tasmax units:", ds_tasmax["tasmax"].attrs.get("units"))
print("pr units    :", ds_pr["pr"].attrs.get("units"))


## 4. Clean

Masks physically-implausible values to NaN. Set `interpolate=True` if you want small gaps filled (default: off — see `clean.py` docstring on why `max_gap` matters).

In [ ]:
ds_tasmax = ci.clean.clean_dataset(ds_tasmax, "tasmax", mask_out_of_range=True, interpolate=False)
ds_pr     = ci.clean.clean_dataset(ds_pr, "pr", mask_out_of_range=True, interpolate=False)


## 5. Compute indices

Set your reference period — this MUST match the actual span of your data.

In [ ]:
ref_start, ref_end = "1985", "2014"   # <-- confirm this matches your data's historical span

temp_indices = ci.indices_temp.compute_temperature_indices(
    ds_tasmax["tasmax"], ref_start=ref_start, ref_end=ref_end
)
precip_indices = ci.indices_precip.compute_precipitation_indices(
    ds_pr["pr"], ref_start=ref_start, ref_end=ref_end
)

print("Temperature indices:", list(temp_indices.data_vars))
print("Precipitation indices:", list(precip_indices.data_vars))


## 6. Save

This is where computation actually triggers (streamed through Dask, not loaded fully into
memory). If this is too slow/memory-heavy in one shot on your real files, save one variable
at a time instead — see the earlier notebook's Section 4 for that fallback pattern.


In [ ]:
all_indices = xr.merge([temp_indices, precip_indices])

encoding = {var: {"zlib": True, "complevel": 4} for var in all_indices.data_vars}
all_indices.to_netcdf("climate_indices_output_v2.nc", encoding=encoding)
print("Saved climate_indices_output_v2.nc")


## 7. Logging (file-based)

Wire up proper file logging before the heavier steps below, so you get a persistent record
of what ran, when, and how long each step took -- not just scrollback in the notebook.


In [ ]:
import logging
import climate_indices as ci

log_path = ci.logging_utils.setup_logging("outputs/logs", run_name="phase2")
logger = logging.getLogger("climate_indices")
print("Logging to:", log_path)


## 8. Statistics

Descriptive stats, climatology, and trend analysis on the indices computed in Section 5.
Wrapped in `timed_step` so duration gets logged automatically.


In [ ]:
with ci.logging_utils.timed_step("Summary statistics", logger):
    txx_summary = ci.stats.summary_stats(all_indices["TXx"])
    rx1day_summary = ci.stats.summary_stats(all_indices["RX1day"])

txx_summary


In [ ]:
with ci.logging_utils.timed_step("Trend analysis (linear + Mann-Kendall)", logger):
    txx_linear_trend = ci.stats.linear_trend(all_indices["TXx"])
    txx_mk_trend = ci.stats.mann_kendall_trend(all_indices["TXx"])

print("Linear trend vars:", list(txx_linear_trend.data_vars))
print("Mann-Kendall vars:", list(txx_mk_trend.data_vars))


**Performance note:** `mann_kendall_trend` and `sens_slope` loop per-grid-cell in Python
(no native vectorization in `pymannkendall`). On your real, full-resolution grid this can be
slow. Test on a coarse subsample first:
```python
txx_sub = all_indices["TXx"].isel(lat=slice(None, None, 5), lon=slice(None, None, 5))
mk_sub = ci.stats.mann_kendall_trend(txx_sub)   # fast sanity check
```
then run the full-resolution version once you trust the logic and are ready to wait.


## 9. Plotting

Every `plotting.*` function returns `(fig, ax)` rather than calling `plt.show()`, so you can
keep customizing before display or saving.


In [ ]:
fig, axes = ci.plotting.plot_multi_map(
    all_indices,
    variables=["TXx", "TX90p", "WSDI", "RX1day", "CDD", "R95p"],
    cmap_map={"TXx": "inferno", "RX1day": "Blues", "CDD": "YlOrBr"},
)


In [ ]:
weights = __import__("numpy").cos(__import__("numpy").deg2rad(all_indices.lat))
weights.name = "weights"

fig, ax = ci.plotting.plot_timeseries(all_indices["TXx"], weights=weights, title="Area-weighted annual TXx")


In [ ]:
fig, ax = ci.plotting.plot_trend_map(
    txx_linear_trend["slope"], txx_linear_trend["p_value"],
    title="TXx trend (°C/year), hatched = p<0.05"
)


In [ ]:
fig, ax = ci.plotting.plot_anomaly_map(
    all_indices["TXx"],
    baseline_start="1985", baseline_end="2014",
    period_start="2065", period_end="2098",
    title="TXx change: far-future minus historical baseline",
)


In [ ]:
fig, ax = ci.plotting.plot_boxplot(all_indices, ["CDD", "CWD"], title="CDD vs CWD spread")


## 10. Export

Write everything to the structured output tree.


In [ ]:
with ci.logging_utils.timed_step("Exporting results", logger):
    dirs = ci.export.make_output_dirs("outputs")

    ci.export.to_netcdf(all_indices, dirs["indices"] / "all_indices.nc")
    ci.export.to_csv(all_indices["TXx"], dirs["statistics"] / "txx_annual_areamean.csv")
    ci.export.to_excel(
        {"TXx": all_indices["TXx"], "RX1day": all_indices["RX1day"], "CDD": all_indices["CDD"]},
        dirs["statistics"] / "index_summary.xlsx",
    )

    fig, ax = ci.plotting.plot_map(all_indices["TXx"].mean(dim="time"), title="Mean annual TXx")
    ci.export.savefig(fig, dirs["plots"] / "txx_climatology.png")

print("Done. See the 'outputs/' folder:")
print("  outputs/indices/all_indices.nc")
print("  outputs/statistics/txx_annual_areamean.csv")
print("  outputs/statistics/index_summary.xlsx")
print("  outputs/plots/txx_climatology.png")
print("  outputs/logs/" + log_path.name)


## Everything built so far

| Module | Status |
|---|---|
| `io_utils.py` | done, tested |
| `validate.py` | done, tested |
| `clean.py` | done, tested (unit-aware range check bug found + fixed) |
| `units.py` | done, tested |
| `indices_temp.py` | done, tested |
| `indices_precip.py` | done, tested |
| `stats.py` | done, tested |
| `plotting.py` | done, tested (11 plot types) |
| `export.py` | done, tested (netcdf/csv/excel/png/geotiff bug found + fixed) |
| `logging_utils.py` | done, tested |

**Not built (deliberately out of scope, per earlier decision):** SPI/SPEI (needs PET),
shapefile/GeoPackage clipping (needs a boundary file), `TNx/TNn/TN90p/TN10p/CSDI/TR/DTR/GSL`
(need `tasmin`/`tas`). Say the word if you get those inputs later and want them added --
same pattern applies (write, test against synthetic data, fix what breaks, then hand it over).

**Also not yet built:** the standalone `.py` script version, `requirements.txt`, and
`README.md` from your original spec (Step 13 deliverables) -- next up if you want them.
